In [1]:
# ---------- Imports ----------
import os
import io
import json
import time
from datetime import date

import numpy as np
import pandas as pd
import requests

# ---------- Display options ----------
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)

# ---------- Library versions (for the attributions slide) ----------
print("numpy   ", np.__version__)
print("pandas  ", pd.__version__)
print("requests", requests.__version__)
from importlib.metadata import version, PackageNotFoundError
for pkg in ("statsmodels", "pyarrow", "matplotlib"):      # used in storage, cleaning and plots
    try:
        print(f"{pkg:9s}", version(pkg))
    except PackageNotFoundError:
        print(f"{pkg:9s} not installed")

numpy    2.4.4
pandas   2.3.3
requests 2.34.2
statsmodels 0.15.0
pyarrow   26.0.0
matplotlib 3.10.8


In [2]:
# ---------- Cities and stations ----------
CITIES = {  # city: (NOAA station id, lat, lon from ghcnd-inventory)
    "Kolkata": ("IN024141500", 22.650, 88.450),
    "Visakhapatnam": ("IN001200200", 17.717, 83.233),
    "Thiruvananthapuram": ("IN010100700", 8.467, 76.950),
}

# ---------- Windows ----------
TEMP_START, TEMP_END = "2022-06-01", "2025-08-31"   # temperature analysis window
RAIN_START, RAIN_END = "2024-01-20", "2025-08-31"   # rain analysis window
YEARS = range(2022, 2026)                           # one Open-Meteo request per year

# ---------- Open-Meteo ----------
MODEL = "gfs_seamless"                              # one model for all lead times
LEADS = range(1, 8)
VARS = ([f"temperature_2m_previous_day{k}" for k in LEADS]
        + [f"precipitation_previous_day{k}" for k in LEADS])
OM_URL = "https://previous-runs-api.open-meteo.com/v1/forecast"
HF_URL = "https://historical-forecast-api.open-meteo.com/v1/forecast"   # check only (section 6 of the notes)
AR_URL = "https://archive-api.open-meteo.com/v1/archive"                # check only (section 6 of the notes)
CHECK_YEAR = 2024                                                        # year used for the check
ACCEPT_OPENMETEO_ROBOTS = True                 # set True after the robots.txt decision

# ---------- NOAA ----------
NOAA_URL = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/"

# ---------- Folders ----------
RAW, INTERIM, CLEAN = "raw", "interim", "clean"
for folder in (RAW, INTERIM, CLEAN):
    os.makedirs(folder, exist_ok=True)

# ---------- Download rules ----------
DELAY = 2                                           # seconds between requests
UA = {"User-Agent": "Mozilla/5.0"}
TODAY = date.today().isoformat()                    # date collected

print("model:", MODEL, "| date collected:", TODAY)
print("cities:", list(CITIES))

model: gfs_seamless | date collected: 2026-11-05
cities: ['Kolkata', 'Visakhapatnam', 'Thiruvananthapuram']


In [3]:
# ---------- Improved download helper ----------
def download(url, path, params=None):

    # Check whether the file is already downloaded
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print("cached :", path)
        return path

    # Try downloading up to 4 times
    for attempt in range(4):

        try:
            r = requests.get(
                url,
                params=params,
                headers=UA,
                timeout=(20, 300)
            )

            # Handle rate limiting
            if r.status_code == 429:
                wait = 30 * (attempt + 1)
                print(f"Rate limited. Waiting {wait}s...")
                time.sleep(wait)
                continue

            r.raise_for_status()

            # Save only after receiving the complete response
            temp_path = path + ".part"

            with open(temp_path, "wb") as f:
                f.write(r.content)

            os.replace(temp_path, path)

            print("saved  :", path, f"({len(r.content):,} bytes)")

            time.sleep(DELAY)

            return path

        except (requests.exceptions.Timeout,
                requests.exceptions.ConnectionError) as e:

            wait = 15 * (attempt + 1)

            print(
                f"Request failed: {e}\n"
                f"Retrying after {wait} seconds..."
            )

            if attempt == 3:
                raise

            time.sleep(wait)

    raise RuntimeError("Download failed after 4 attempts: " + url)

In [4]:
# ---------- Source table ----------
rows = [
    {"source": "NOAA GHCN-Daily", "what": "station list",
     "url": NOAA_URL + "ghcnd-stations.txt", "filters": "none: whole file"},
    {"source": "NOAA GHCN-Daily", "what": "station inventory",
     "url": NOAA_URL + "ghcnd-inventory.txt", "filters": "none: whole file"},
]
for city, (sid, lat, lon) in CITIES.items():
    rows.append({"source": "NOAA GHCN-Daily", "what": f"daily observations, {city}",
                 "url": NOAA_URL + f"by_station/{sid}.csv.gz",
                 "filters": f"station {sid}; whole file (all dates, all elements); window and elements are filtered in cleaning"})
rows.append({"source": "Open-Meteo Previous Runs API",
             "what": f"hourly forecasts, model {MODEL}, 3 cities x {len(YEARS)} years",
             "url": OM_URL,
             "filters": (f"one request per city and year; latitude and longitude = station coordinates; "
                         f"start_date {YEARS[0]}-01-01 to end_date {YEARS[-1]}-12-31 (one year per request); "
                         f"models={MODEL}; timezone=Asia/Kolkata; hourly = {len(VARS)} variables "
                         f"(temperature_2m_previous_day1..7 and precipitation_previous_day1..7)")})
rows.append({"source": "Open-Meteo Historical Forecast API",
             "what": f"check only: hourly temperature_2m, model {MODEL}, {CHECK_YEAR}, 3 cities",
             "url": HF_URL,
             "filters": (f"latitude and longitude = station coordinates; start_date {CHECK_YEAR}-01-01 to "
                         f"end_date {CHECK_YEAR}-12-31; models={MODEL}; timezone=Asia/Kolkata; hourly=temperature_2m")})
rows.append({"source": "Open-Meteo Archive API",
             "what": f"check only: hourly temperature_2m, {CHECK_YEAR}, 3 cities",
             "url": AR_URL,
             "filters": (f"latitude and longitude = station coordinates; start_date {CHECK_YEAR}-01-01 to "
                         f"end_date {CHECK_YEAR}-12-31; timezone=Asia/Kolkata; hourly=temperature_2m")})

# ---------- Add date collected and licence ----------
sources = pd.DataFrame(rows)
sources["date_collected"] = TODAY
sources["licence"] = np.where(
    sources["source"].str.startswith("NOAA"),
    "CC0 1.0 (NOAA GHCN-Daily)",
    "CC BY 4.0 (Open-Meteo API)")

# ---------- Show and save ----------
print(sources[["source", "what", "url", "date_collected"]].to_string(index=False))
print("\nfilters used:")
for _, row in sources.iterrows():
    print(f"- {row['what']}: {row['filters']}")
sources.to_csv(f"{INTERIM}/sources_{TODAY}.csv", index=False)

                            source                                                                  what                                                                         url date_collected
                   NOAA GHCN-Daily                                                          station list            https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt     2026-11-05
                   NOAA GHCN-Daily                                                     station inventory           https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-inventory.txt     2026-11-05
                   NOAA GHCN-Daily                                           daily observations, Kolkata https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_station/IN024141500.csv.gz     2026-11-05
                   NOAA GHCN-Daily                                     daily observations, Visakhapatnam https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_station/IN001200200.csv.gz     2026-11-05
                   N

In [5]:
# ---------- Sites and the paths we use ----------
sites = {
    "noaa": ("https://www.ncei.noaa.gov/robots.txt",
             "/pub/data/ghcn/daily/by_station/IN024141500.csv.gz"),
    "openmeteo": ("https://previous-runs-api.open-meteo.com/robots.txt",
                  "/v1/forecast"),
}

# ---------- Download, print and check each robots.txt ----------
for name, (url, our_path) in sites.items():
    path = download(url, f"{RAW}/robots_{name}_{TODAY}.txt")
    with open(path, encoding="utf-8") as f:
        text = f.read()

    print(f"\n=== robots.txt: {url} ({TODAY}) ===")
    print(text)

    # ---------- Compare our path with each Disallow rule ----------
    rules = [line.split(":", 1)[1].strip()
             for line in text.splitlines() if line.lower().startswith("disallow:")]
    blocked = [r for r in rules if r and our_path.startswith(r.rstrip("*"))]
    print("our path   :", our_path)
    print("blocked by :", blocked if blocked else "no rule matches")

saved  : raw/robots_noaa_2026-11-05.txt (254 bytes)

=== robots.txt: https://www.ncei.noaa.gov/robots.txt (2026-11-05) ===
User-agent: *
Disallow: /data*
Disallow: /orders*

Allow: /data/oceans/coris/library*

#CRN
Disallow: /access/crn/*?
Disallow: /access/crn/ws/
Disallow: /access/crn/flex/
Disallow: /access/crn/error/
Disallow: /access/crn/new*
Disallow: /access/crn/xml*

our path   : /pub/data/ghcn/daily/by_station/IN024141500.csv.gz
blocked by : no rule matches
saved  : raw/robots_openmeteo_2026-11-05.txt (25 bytes)

=== robots.txt: https://previous-runs-api.open-meteo.com/robots.txt (2026-11-05) ===
User-agent: *
Disallow: /
our path   : /v1/forecast
blocked by : ['/']


In [6]:
# ---------- NOAA file list: name in raw/ and URL ----------
noaa_files = {
    "stations": (f"{RAW}/noaa_ghcnd-stations_{TODAY}.txt", NOAA_URL + "ghcnd-stations.txt"),
    "inventory": (f"{RAW}/noaa_ghcnd-inventory_{TODAY}.txt", NOAA_URL + "ghcnd-inventory.txt"),
}
for city, (sid, lat, lon) in CITIES.items():
    noaa_files[city] = (f"{RAW}/noaa_ghcnd_{sid}_{TODAY}.csv.gz",
                        NOAA_URL + f"by_station/{sid}.csv.gz")

# ---------- Download (cached, 2 s delay) ----------
noaa_paths = {}
for key, (path, url) in noaa_files.items():
    noaa_paths[key] = download(url, path)

# ---------- Check: files exist and are not empty ----------
for key, path in noaa_paths.items():
    print(f"{key:20s} {os.path.getsize(path):>12,} bytes  {path}")

saved  : raw/noaa_ghcnd-stations_2026-11-05.txt (11,441,526 bytes)
saved  : raw/noaa_ghcnd-inventory_2026-11-05.txt (36,402,100 bytes)
saved  : raw/noaa_ghcnd_IN024141500_2026-11-05.csv.gz (292,044 bytes)
saved  : raw/noaa_ghcnd_IN001200200_2026-11-05.csv.gz (169,672 bytes)
saved  : raw/noaa_ghcnd_IN010100700_2026-11-05.csv.gz (96,415 bytes)
stations               11,441,526 bytes  raw/noaa_ghcnd-stations_2026-11-05.txt
inventory              36,402,100 bytes  raw/noaa_ghcnd-inventory_2026-11-05.txt
Kolkata                   292,044 bytes  raw/noaa_ghcnd_IN024141500_2026-11-05.csv.gz
Visakhapatnam             169,672 bytes  raw/noaa_ghcnd_IN001200200_2026-11-05.csv.gz
Thiruvananthapuram         96,415 bytes  raw/noaa_ghcnd_IN010100700_2026-11-05.csv.gz


In [7]:
# ---------- Open-Meteo calls (3 cities x 4 years = 12 requests) ----------
om_paths = {}

if not ACCEPT_OPENMETEO_ROBOTS:
    print("Flag is False. Set ACCEPT_OPENMETEO_ROBOTS = True in block 2, then re-run blocks 2 and 7.")
else:
    for city, (sid, lat, lon) in CITIES.items():
        for y in YEARS:
            # ---------- Request settings for one city and one year ----------
            params = {"latitude": lat, "longitude": lon, "timezone": "Asia/Kolkata",
                      "start_date": f"{y}-01-01", "end_date": f"{y}-12-31",
                      "hourly": ",".join(VARS), "models": MODEL}
            path = f"{RAW}/openmeteo_{MODEL}_{city}_{y}_{TODAY}.json"

            # ---------- Download (cached, 2 s delay) ----------
            om_paths[(city, y)] = download(OM_URL, path, params)

    # ---------- Check: files exist and are not empty ----------
    for (city, y), path in om_paths.items():
        print(f"{city:20s} {y}  {os.path.getsize(path):>10,} bytes")

saved  : raw/openmeteo_gfs_seamless_Kolkata_2022_2026-11-05.json (780,838 bytes)
saved  : raw/openmeteo_gfs_seamless_Kolkata_2023_2026-11-05.json (780,838 bytes)
saved  : raw/openmeteo_gfs_seamless_Kolkata_2024_2026-11-05.json (783,053 bytes)
saved  : raw/openmeteo_gfs_seamless_Kolkata_2025_2026-11-05.json (780,863 bytes)
saved  : raw/openmeteo_gfs_seamless_Visakhapatnam_2022_2026-11-05.json (780,839 bytes)
saved  : raw/openmeteo_gfs_seamless_Visakhapatnam_2023_2026-11-05.json (780,839 bytes)
saved  : raw/openmeteo_gfs_seamless_Visakhapatnam_2024_2026-11-05.json (782,999 bytes)
saved  : raw/openmeteo_gfs_seamless_Visakhapatnam_2025_2026-11-05.json (780,869 bytes)
saved  : raw/openmeteo_gfs_seamless_Thiruvananthapuram_2022_2026-11-05.json (780,838 bytes)
saved  : raw/openmeteo_gfs_seamless_Thiruvananthapuram_2023_2026-11-05.json (780,839 bytes)
saved  : raw/openmeteo_gfs_seamless_Thiruvananthapuram_2024_2026-11-05.json (783,048 bytes)
saved  : raw/openmeteo_gfs_seamless_Thiruvananthapur

In [8]:
# ---------- Check from the project notes (section 6): do the Open-Meteo historical-forecast and archive APIs return the same values? ----------
# If both give the same hourly values, neither carries a lead time, so the Previous Runs API (leads 1 to 7)
# is the right source for this project. One year, three cities, temperature only.
check_paths = {}

if not ACCEPT_OPENMETEO_ROBOTS:
    print("Flag is False. Check skipped.")
else:
    # ---------- robots.txt of the two extra hosts (saved in raw/, quote in the report) ----------
    for name, url in (("openmeteo-histforecast", HF_URL.split("/v1")[0] + "/robots.txt"),
                      ("openmeteo-archive", AR_URL.split("/v1")[0] + "/robots.txt")):
        try:
            p = download(url, f"{RAW}/robots_{name}_{TODAY}.txt")
            with open(p, encoding="utf-8") as f:
                print(f"\n=== robots.txt: {url} ({TODAY}) ===\n{f.read()}")
        except requests.HTTPError as e:
            print(f"\n=== robots.txt: {url} -> {e} (no file saved) ===")

    # ---------- Same place, same hours, temperature only ----------
    for city, (sid, lat, lon) in CITIES.items():
        common = {"latitude": lat, "longitude": lon, "timezone": "Asia/Kolkata",
                  "start_date": f"{CHECK_YEAR}-01-01", "end_date": f"{CHECK_YEAR}-12-31",
                  "hourly": "temperature_2m"}
        check_paths[(city, "histforecast")] = download(
            HF_URL, f"{RAW}/openmeteo_histforecast_{MODEL}_{city}_{CHECK_YEAR}_{TODAY}.json",
            {**common, "models": MODEL})
        check_paths[(city, "archive")] = download(
            AR_URL, f"{RAW}/openmeteo_archive_{city}_{CHECK_YEAR}_{TODAY}.json", common)

saved  : raw/robots_openmeteo-histforecast_2026-11-05.txt (25 bytes)

=== robots.txt: https://historical-forecast-api.open-meteo.com/robots.txt (2026-11-05) ===
User-agent: *
Disallow: /
saved  : raw/robots_openmeteo-archive_2026-11-05.txt (25 bytes)

=== robots.txt: https://archive-api.open-meteo.com/robots.txt (2026-11-05) ===
User-agent: *
Disallow: /
saved  : raw/openmeteo_histforecast_gfs_seamless_Kolkata_2024_2026-11-05.json (211,099 bytes)
saved  : raw/openmeteo_archive_Kolkata_2024_2026-11-05.json (211,096 bytes)
saved  : raw/openmeteo_histforecast_gfs_seamless_Visakhapatnam_2024_2026-11-05.json (211,099 bytes)
saved  : raw/openmeteo_archive_Visakhapatnam_2024_2026-11-05.json (211,098 bytes)
saved  : raw/openmeteo_histforecast_gfs_seamless_Thiruvananthapuram_2024_2026-11-05.json (211,099 bytes)
saved  : raw/openmeteo_archive_Thiruvananthapuram_2024_2026-11-05.json (211,099 bytes)


In [9]:
# ---------- Compare: hourly agreement between the sources, then daily TMAX against the station ----------
assert check_paths, "run the previous cell with ACCEPT_OPENMETEO_ROBOTS = True first"

def hourly_series(path, prefix):
    with open(path, encoding="utf-8") as f:
        h = json.load(f)["hourly"]
    key = next(k for k in h if k.startswith(prefix))
    return pd.Series(pd.to_numeric(pd.Series(h[key]), errors="coerce").values,
                     index=pd.to_datetime(h["time"]))

def daily_tmax(s):
    return s.resample("D").max().where(s.resample("D").count() == 24)      # same rule as in cleaning

def agreement(a, b):
    both = pd.concat([a, b], axis=1, keys=["a", "b"]).dropna()
    diff = (both["a"] - both["b"]).abs()
    return {"hours": len(both), "identical_pct": round(100 * (diff < 0.05).mean(), 1),
            "mean_abs_diff_C": round(diff.mean(), 2)}

st_cols = ["id", "date", "element", "value", "m_flag", "q_flag", "s_flag", "obs_time"]
agree_rows, mae_rows = [], []
for city in CITIES:
    hf = hourly_series(check_paths[(city, "histforecast")], "temperature_2m")
    ar = hourly_series(check_paths[(city, "archive")], "temperature_2m")
    d1 = hourly_series(om_paths[(city, CHECK_YEAR)], "temperature_2m_previous_day1")
    d7 = hourly_series(om_paths[(city, CHECK_YEAR)], "temperature_2m_previous_day7")

    pairs = {"historical forecast vs archive": (hf, ar),
             "historical forecast vs previous run, lead 1": (hf, d1),
             "historical forecast vs previous run, lead 7": (hf, d7)}
    for label, (a, b) in pairs.items():
        agree_rows.append({"city": city, "comparison": label, **agreement(a, b)})

    # ---------- Station TMAX for the same year (quality-flag rows removed, tenths to deg C) ----------
    st = pd.read_csv(noaa_paths[city], header=None, names=st_cols, low_memory=False)
    st = st[(st["element"] == "TMAX") & st["q_flag"].isna()].copy()
    st["date"] = pd.to_datetime(st["date"].astype(str), format="%Y%m%d")
    st = st[st["date"].dt.year == CHECK_YEAR].set_index("date")["value"] / 10

    for label, s in {"historical forecast": hf, "archive": ar,
                     "previous run, lead 1": d1, "previous run, lead 7": d7}.items():
        both = pd.concat([daily_tmax(s), st], axis=1, keys=["fc", "obs"]).dropna()
        mae_rows.append({"city": city, "series": label, "days": len(both),
                         "MAE_vs_station_C": round((both["fc"] - both["obs"]).abs().mean(), 2)})

agree, mae = pd.DataFrame(agree_rows), pd.DataFrame(mae_rows)
print(f"hourly agreement between the sources, {CHECK_YEAR}:")
print(agree.to_string(index=False))
print(f"\ndaily TMAX against the station, {CHECK_YEAR}:")
print(mae.to_string(index=False))
pd.concat([agree.assign(table="agreement"), mae.assign(table="mae_vs_station")]).to_csv(
    f"{INTERIM}/api_check_{TODAY}.csv", index=False)
# Reading: if "historical forecast vs archive" is identical in (nearly) every hour, the two APIs carry no
# lead time. Only the Previous Runs columns (lead 1 to 7) can show how error grows with lead time.

hourly agreement between the sources, 2024:
              city                                  comparison  hours  identical_pct  mean_abs_diff_C
           Kolkata              historical forecast vs archive   8784            2.2             1.50
           Kolkata historical forecast vs previous run, lead 1   8335            5.3             0.80
           Kolkata historical forecast vs previous run, lead 7   8191            2.5             1.67
     Visakhapatnam              historical forecast vs archive   8784            3.2             1.19
     Visakhapatnam historical forecast vs previous run, lead 1   8335            8.1             0.65
     Visakhapatnam historical forecast vs previous run, lead 7   8191            2.8             1.36
Thiruvananthapuram              historical forecast vs archive   8784            2.5             1.40
Thiruvananthapuram historical forecast vs previous run, lead 1   8335            8.8             0.53
Thiruvananthapuram historical forecast

In [10]:
# ---------- Profile helper: shape, dtypes, meaning of each column ----------
def profile(name, df, meaning):
    print(f"\n=== {name} ===")
    print("shape:", df.shape)
    info = pd.DataFrame({"dtype": df.dtypes.astype(str),
                         "non_null": df.notna().sum(),
                         "meaning": pd.Series(meaning)})
    print(info.to_string())
    print(df.head(3).to_string(index=False))

# ---------- Station files: 8 columns, no header row ----------
cols = ["id", "date", "element", "value", "m_flag", "q_flag", "s_flag", "obs_time"]
meaning = {
    "id": "station ID (GHCN code)",
    "date": "observation date as number YYYYMMDD",
    "element": "TMAX, TMIN, PRCP, ... (what was measured)",
    "value": "TMAX/TMIN in tenths of deg C, PRCP in tenths of mm",
    "m_flag": "measurement flag (blank = none)",
    "q_flag": "quality flag (blank = passed quality check)",
    "s_flag": "source flag (which data source)",
    "obs_time": "observation time HHMM (mostly empty)",
}
for city, (sid, lat, lon) in CITIES.items():
    df = pd.read_csv(noaa_paths[city], header=None, names=cols)   # raw, nothing changed
    profile(f"{city} ({sid})", df, meaning)
    print("elements:", df["element"].value_counts().head(6).to_dict())

# ---------- Inventory file: 6 columns split by spaces ----------
inv = pd.read_csv(noaa_paths["inventory"], sep=r"\s+", header=None,
                  names=["id", "lat", "lon", "element", "first_year", "last_year"])
profile("ghcnd-inventory", inv, {
    "id": "station ID", "lat": "latitude (deg N)", "lon": "longitude (deg E)",
    "element": "what the station measures", "first_year": "first year with data",
    "last_year": "last year with data"})

# ---------- Station list: fixed-width text, read as one text column ----------
st = pd.read_csv(noaa_paths["stations"], sep="\t", header=None, names=["line"], quoting=3)
print("\n=== ghcnd-stations ===")
print("shape:", st.shape, "| dtype:", st["line"].dtype)
print("one line = one station, columns by character position:")
print("  0-11 id | 12-20 lat | 21-30 lon | 31-37 elevation (m) | 38-40 state | 41-71 name | rest flags/WMO id")
mine = st[st["line"].str[0:11].isin([v[0] for v in CITIES.values()])]
for line in mine["line"]:
    print(line)


=== Kolkata (IN024141500) ===
shape: (67405, 8)
            dtype  non_null                                             meaning
id         object     67405                              station ID (GHCN code)
date        int64     67405                 observation date as number YYYYMMDD
element    object     67405           TMAX, TMIN, PRCP, ... (what was measured)
value       int64     67405  TMAX/TMIN in tenths of deg C, PRCP in tenths of mm
m_flag     object     24928                     measurement flag (blank = none)
q_flag     object         5         quality flag (blank = passed quality check)
s_flag     object     67405                     source flag (which data source)
obs_time  float64         0                observation time HHMM (mostly empty)
         id     date element  value m_flag q_flag s_flag  obs_time
IN024141500 19430831    TAVG    269      H    NaN      S       NaN
IN024141500 19430901    TAVG    282      H    NaN      S       NaN
IN024141500 19430902    TAVG  

C:\Users\Hari Kaeilash G\AppData\Local\Temp\ipykernel_29144\1245243218.py:24: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(noaa_paths[city], header=None, names=cols)   # raw, nothing changed



=== Thiruvananthapuram (IN010100700) ===
shape: (23757, 8)
            dtype  non_null                                             meaning
id         object     23757                              station ID (GHCN code)
date        int64     23757                 observation date as number YYYYMMDD
element    object     23757           TMAX, TMIN, PRCP, ... (what was measured)
value       int64     23757  TMAX/TMIN in tenths of deg C, PRCP in tenths of mm
m_flag     object     10151                     measurement flag (blank = none)
q_flag     object         1         quality flag (blank = passed quality check)
s_flag     object     23757                     source flag (which data source)
obs_time  float64         0                observation time HHMM (mostly empty)
         id     date element  value m_flag q_flag s_flag  obs_time
IN010100700 19550813    PRCP      0    NaN    NaN      I       NaN
IN010100700 19550814    PRCP      0    NaN    NaN      I       NaN
IN010100700 1955081

In [11]:
# ---------- Meaning of each hourly column ----------
meaning = {"time": "hour start, local time Asia/Kolkata, text YYYY-MM-DDTHH:MM"}
for k in LEADS:
    meaning[f"temperature_2m_previous_day{k}"] = f"air temperature at 2 m (deg C), forecast made {k} day(s) before"
    meaning[f"precipitation_previous_day{k}"] = f"rain in the past hour (mm), forecast made {k} day(s) before"

# ---------- Read every raw JSON file, nothing changed ----------
om_raw = {}
for (city, y), path in om_paths.items():
    with open(path, encoding="utf-8") as f:
        om_raw[(city, y)] = json.load(f)

# ---------- Full profile of the first file ----------
first = list(om_raw)[0]
js = om_raw[first]
print("top-level keys:", list(js))
print("metadata:", {k: v for k, v in js.items() if k not in ("hourly", "hourly_units")})
print("units:", js["hourly_units"])
df = pd.DataFrame(js["hourly"])
profile(f"{first[0]} {first[1]}", df, {c: meaning.get(c.replace("_" + MODEL, ""), "") for c in df.columns})

# ---------- Shape and missing count for all files ----------
rows = []
for (city, y), js in om_raw.items():
    d = pd.DataFrame(js["hourly"])
    rows.append({"city": city, "year": y, "rows": d.shape[0], "columns": d.shape[1],
                 "grid_elevation_m": js["elevation"], "null_cells": int(d.isna().sum().sum())})
print()
print(pd.DataFrame(rows).to_string(index=False))

top-level keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
metadata: {'latitude': 22.668404, 'longitude': 88.47656, 'generationtime_ms': 35.55643558502197, 'utc_offset_seconds': 19800, 'timezone': 'Asia/Kolkata', 'timezone_abbreviation': 'GMT+5:30', 'elevation': 6.0}
units: {'time': 'iso8601', 'temperature_2m_previous_day1': '°C', 'temperature_2m_previous_day2': '°C', 'temperature_2m_previous_day3': '°C', 'temperature_2m_previous_day4': '°C', 'temperature_2m_previous_day5': '°C', 'temperature_2m_previous_day6': '°C', 'temperature_2m_previous_day7': '°C', 'precipitation_previous_day1': 'mm', 'precipitation_previous_day2': 'mm', 'precipitation_previous_day3': 'mm', 'precipitation_previous_day4': 'mm', 'precipitation_previous_day5': 'mm', 'precipitation_previous_day6': 'mm', 'precipitation_previous_day7': 'mm'}

=== Kolkata 2022 ===
shape: (8760, 15)
                                dtype 

In [12]:
# ---------- Folders ----------
for folder in (RAW, INTERIM, CLEAN):
    print(f"{folder:8s} exists: {os.path.isdir(folder)}  ->  {os.path.abspath(folder)}")

# ---------- File list for raw/ ----------
files = pd.DataFrame({"file": sorted(os.listdir(RAW))})
files["size_bytes"] = [os.path.getsize(f"{RAW}/{f}") for f in files["file"]]

# ---------- Naming check: source first, date collected in the name ----------
files["source"] = files["file"].str.split("_").str[0]
files["date_in_name"] = files["file"].str.extract(r"(\d{4}-\d{2}-\d{2})")
files["name_ok"] = (files["source"].isin(["noaa", "openmeteo", "robots"])
                    & files["date_in_name"].notna())          # any valid date; files from earlier runs are fine
print()
print(files.to_string(index=False))
print("\nfiles with a bad name:", int((~files["name_ok"]).sum()))

# ---------- Save the file list (a manifest of raw/) ----------
files.to_csv(f"{INTERIM}/raw_manifest_{TODAY}.csv", index=False)
print("raw files are only read after this point, never edited")

raw      exists: True  ->  c:\Users\Hari Kaeilash G\OneDrive\DCV_project_final\raw
interim  exists: True  ->  c:\Users\Hari Kaeilash G\OneDrive\DCV_project_final\interim
clean    exists: True  ->  c:\Users\Hari Kaeilash G\OneDrive\DCV_project_final\clean

                                                                       file  size_bytes    source date_in_name  name_ok
                                        noaa_ghcnd-inventory_2026-11-05.txt    36402100      noaa   2026-11-05     True
                                         noaa_ghcnd-stations_2026-11-05.txt    11441526      noaa   2026-11-05     True
                                   noaa_ghcnd_IN001200200_2026-11-05.csv.gz      169672      noaa   2026-11-05     True
                                   noaa_ghcnd_IN010100700_2026-11-05.csv.gz       96415      noaa   2026-11-05     True
                                   noaa_ghcnd_IN024141500_2026-11-05.csv.gz      292044      noaa   2026-11-05     True
                        

In [13]:
# ---------- NOAA: all station rows in one table (parsed, nothing dropped) ----------
cols = ["id", "date", "element", "value", "m_flag", "q_flag", "s_flag", "obs_time"]
parts = []
for city, (sid, lat, lon) in CITIES.items():
    d = pd.read_csv(noaa_paths[city], header=None, names=cols)
    d["city"] = city
    parts.append(d)
noaa = pd.concat(parts, ignore_index=True)

# ---------- NOAA: fix types only ----------
noaa["date"] = pd.to_datetime(noaa["date"].astype(str), format="%Y%m%d")
null_before = int(noaa["value"].isna().sum())
noaa["value"] = pd.to_numeric(noaa["value"], errors="coerce")
print("NOAA value: text values turned into missing:", int(noaa["value"].isna().sum()) - null_before)
print("NOAA rows per city:")
print(noaa.groupby("city").size().to_string())

# ---------- Open-Meteo: all hourly rows in one table (parsed, nothing dropped) ----------
parts = []
for (city, y), js in om_raw.items():
    d = pd.DataFrame(js["hourly"])
    d.columns = [c.replace("_" + MODEL, "") for c in d.columns]
    d.insert(0, "city", city)
    d.insert(1, "model", MODEL)
    parts.append(d)
om = pd.concat(parts, ignore_index=True)

# ---------- Open-Meteo: fix types only ----------
om["time"] = pd.to_datetime(om["time"])
num_cols = list(om.columns[3:])
n_object = int(sum(om[c].dtype == object for c in num_cols))      # before
null_before = int(om[num_cols].isna().sum().sum())
for c in num_cols:
    om[c] = pd.to_numeric(om[c], errors="coerce")
null_after = int(om[num_cols].isna().sum().sum())
print(f"\nOpen-Meteo number columns stored as object before: {n_object} of {len(num_cols)}"
      f" | float64 after: {int(sum(om[c].dtype == 'float64' for c in num_cols))} of {len(num_cols)}")
print(f"missing cells before: {null_before} | after: {null_after} | text values lost: {null_after - null_before}")
print("\nOpen-Meteo rows per city:")
print(om.groupby("city").size().to_string())

# ---------- Save to interim/ (source and date in the name) ----------
noaa_file = f"{INTERIM}/noaa_ghcnd_3stations_{TODAY}.csv"
om_file = f"{INTERIM}/openmeteo_{MODEL}_hourly_{TODAY}.csv"
noaa.to_csv(noaa_file, index=False)
om.to_csv(om_file, index=False)

# ---------- Read back and compare shape ----------
noaa_back = pd.read_csv(noaa_file, parse_dates=["date"])
om_back = pd.read_csv(om_file, parse_dates=["time"])
print("\nNOAA :", noaa.shape, "->", noaa_back.shape, "| same:", noaa.shape == noaa_back.shape)
print("Meteo:", om.shape, "->", om_back.shape, "| same:", om.shape == om_back.shape)

C:\Users\Hari Kaeilash G\AppData\Local\Temp\ipykernel_29144\3804128593.py:5: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  d = pd.read_csv(noaa_paths[city], header=None, names=cols)


NOAA value: text values turned into missing: 0
NOAA rows per city:
city
Kolkata               67405
Thiruvananthapuram    23757
Visakhapatnam         46025


C:\Users\Hari Kaeilash G\AppData\Local\Temp\ipykernel_29144\3804128593.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  om = pd.concat(parts, ignore_index=True)



Open-Meteo number columns stored as object before: 0 of 14 | float64 after: 14 of 14
missing cells before: 390705 | after: 390705 | text values lost: 0

Open-Meteo rows per city:
city
Kolkata               35064
Thiruvananthapuram    35064
Visakhapatnam         35064


C:\Users\Hari Kaeilash G\AppData\Local\Temp\ipykernel_29144\3804128593.py:49: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  noaa_back = pd.read_csv(noaa_file, parse_dates=["date"])



NOAA : (137187, 9) -> (137187, 9) | same: True
Meteo: (105192, 17) -> (105192, 17) | same: True


In [14]:
# ---------- Helper: one dictionary row per column ----------
def dict_rows(table_name, df, info):
    out = []
    for col in df.columns:
        unit, source, allowed = info[col]
        s = df[col]
        numeric_or_date = pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s)
        out.append({"table": table_name, "column": col, "dtype": str(s.dtype),
                    "unit": unit, "source": source, "allowed_range": allowed,
                    "observed_min": s.min() if numeric_or_date else "",
                    "observed_max": s.max() if numeric_or_date else ""})
    return out

# ---------- NOAA columns: unit, source, allowed range ----------
S1 = "NOAA GHCN-Daily"
noaa_info = {
    "id": ("-", S1, "3 station IDs in CITIES"),
    "date": ("date", S1, "2022-01-01 to 2025-08-31"),
    "element": ("-", S1, "TMAX, TMIN, PRCP, TAVG, ..."),
    "value": ("tenths of deg C (TMAX/TMIN), tenths of mm (PRCP)", S1,
              "TMAX -100 to 600; PRCP 0 to 10000"),
    "m_flag": ("-", S1, "blank or one letter"),
    "q_flag": ("-", S1, "blank = passed quality check"),
    "s_flag": ("-", S1, "one character"),
    "obs_time": ("HHMM", S1, "0000 to 2359 or blank"),
    "city": ("-", "added by us", "Kolkata, Visakhapatnam, Thiruvananthapuram"),
}

# ---------- Open-Meteo columns: unit, source, allowed range ----------
S2 = "Open-Meteo Previous Runs API"
om_info = {
    "city": ("-", "added by us", "3 cities"),
    "model": ("-", "added by us", MODEL),
    "time": ("hour, local time Asia/Kolkata", S2, "2022-01-01 to 2025-12-31"),
}
for k in LEADS:
    om_info[f"temperature_2m_previous_day{k}"] = ("deg C", S2, "-10 to 55")
    om_info[f"precipitation_previous_day{k}"] = ("mm per hour", S2, "0 to 200")

# ---------- Build, show and save the dictionary ----------
dictionary = pd.DataFrame(dict_rows("noaa_ghcnd_3stations", noaa, noaa_info)
                          + dict_rows("openmeteo_hourly", om, om_info))
print(dictionary.to_string(index=False))
dictionary.to_csv(f"{INTERIM}/data_dictionary_draft_{TODAY}.csv", index=False)

# ---------- NOAA value range per element (mixed units, so check by element) ----------
print("\nNOAA value by element (tenths):")
print(noaa[noaa["element"].isin(["TMAX", "TMIN", "PRCP"])]
      .groupby("element")["value"].agg(["min", "max", "count"]).to_string())

               table                       column          dtype                                             unit                       source                              allowed_range        observed_min        observed_max
noaa_ghcnd_3stations                           id         object                                                -              NOAA GHCN-Daily                    3 station IDs in CITIES                                        
noaa_ghcnd_3stations                         date datetime64[ns]                                             date              NOAA GHCN-Daily                   2022-01-01 to 2025-08-31 1901-01-01 00:00:00 2025-08-24 00:00:00
noaa_ghcnd_3stations                      element         object                                                -              NOAA GHCN-Daily                TMAX, TMIN, PRCP, TAVG, ...                                        
noaa_ghcnd_3stations                        value          int64 tenths of deg C (TMAX/TMIN), te

In [15]:
# ---------- Save both tables as Parquet (needs: pip install pyarrow) ----------
noaa_pq = f"{INTERIM}/noaa_ghcnd_3stations_{TODAY}.parquet"
om_pq = f"{INTERIM}/openmeteo_{MODEL}_hourly_{TODAY}.parquet"
noaa.to_parquet(noaa_pq, index=False)
om.to_parquet(om_pq, index=False)

# ---------- Reason for the format: file size, CSV vs Parquet ----------
sizes = pd.DataFrame({
    "csv_MB": [os.path.getsize(noaa_file) / 1e6, os.path.getsize(om_file) / 1e6],
    "parquet_MB": [os.path.getsize(noaa_pq) / 1e6, os.path.getsize(om_pq) / 1e6],
}, index=["noaa", "openmeteo"]).round(2)
print(sizes.to_string())

# ---------- Read back and check dtypes ----------
def check_dtypes(name, original, path):
    back = pd.read_parquet(path)
    cmp = pd.DataFrame({"before": original.dtypes.astype(str),
                        "after": back.dtypes.astype(str)})
    cmp["same"] = cmp["before"] == cmp["after"]
    print(f"\n=== {name}: shape {original.shape} -> {back.shape} ===")
    print(cmp.to_string())
    print("all dtypes same:", bool(cmp["same"].all()),
          "| shape same:", original.shape == back.shape)

check_dtypes("noaa", noaa, noaa_pq)
check_dtypes("openmeteo", om, om_pq)

# ---------- Same check for the CSV read back (without parse_dates) ----------
print("\nCSV read back without parse_dates, date column dtype:",
      pd.read_csv(noaa_file)["date"].dtype)

           csv_MB  parquet_MB
noaa         6.80        0.75
openmeteo   10.46        1.38

=== noaa: shape (137187, 9) -> (137187, 9) ===
                  before           after  same
id                object          object  True
date      datetime64[ns]  datetime64[ns]  True
element           object          object  True
value              int64           int64  True
m_flag            object          object  True
q_flag            object          object  True
s_flag            object          object  True
obs_time         float64         float64  True
city              object          object  True
all dtypes same: True | shape same: True

=== openmeteo: shape (105192, 17) -> (105192, 17) ===
                                      before           after  same
city                                  object          object  True
model                                 object          object  True
time                          datetime64[ns]  datetime64[ns]  True
temperature_2m_previous_day1

C:\Users\Hari Kaeilash G\AppData\Local\Temp\ipykernel_29144\1267073664.py:30: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(noaa_file)["date"].dtype)


In [16]:
# ---------- Duplicate check (nothing is removed here) ----------
# NOAA: exact duplicate rows and duplicate key (city, date, element)
print("NOAA rows:", len(noaa))
print("  exact duplicate rows :", int(noaa.duplicated().sum()))
print("  duplicate key        :", int(noaa.duplicated(["city", "date", "element"]).sum()))

# Open-Meteo: exact duplicate rows and duplicate key (city, time)
print("\nOpen-Meteo rows:", len(om))
print("  exact duplicate rows :", int(om.duplicated().sum()))
print("  duplicate key        :", int(om.duplicated(["city", "time"]).sum()))

# ---------- Show a few duplicate-key rows, if any ----------
dup_noaa = noaa[noaa.duplicated(["city", "date", "element"], keep=False)]
dup_om = om[om.duplicated(["city", "time"], keep=False)]
print("\nNOAA rows involved:", len(dup_noaa), "| Open-Meteo rows involved:", len(dup_om))
if len(dup_noaa):
    print(dup_noaa.sort_values(["city", "date", "element"]).head(6).to_string(index=False))
if len(dup_om):
    print(dup_om.sort_values(["city", "time"]).head(6).iloc[:, :5].to_string(index=False))

NOAA rows: 137187
  exact duplicate rows : 0
  duplicate key        : 0

Open-Meteo rows: 105192
  exact duplicate rows : 0
  duplicate key        : 0

NOAA rows involved: 0 | Open-Meteo rows involved: 0


In [17]:
# ---------- Forecast side: first and last complete day per variable ----------
rows = []
for city, g in om.groupby("city"):
    g = g.set_index("time")
    for col in ["temperature_2m_previous_day1", "temperature_2m_previous_day7",
                "precipitation_previous_day1", "precipitation_previous_day7"]:
        n = g[col].resample("D").count()
        full = n[n == 24].index                       # days with all 24 hours
        rows.append({"city": city, "variable": col,
                     "first_day": full.min().date(), "last_day": full.max().date()})
print(pd.DataFrame(rows).to_string(index=False))

# ---------- Station side: last day with a TMAX or PRCP value ----------
last = (noaa[noaa["element"].isin(["TMAX", "PRCP"])]
        .groupby(["city", "element"])["date"].max().dt.date)
print("\nlast station record:")
print(last.to_string())

# ---------- Window length and monsoon days (June to September) ----------
def window_info(name, start, end):
    d = pd.Series(pd.date_range(start, end))
    m = d[d.dt.month.isin([6, 7, 8, 9])]
    print(f"\n{name}: {start} to {end} = {len(d)} days")
    print("monsoon days per year:", m.groupby(m.dt.year).size().to_dict())

window_info("temperature", TEMP_START, TEMP_END)
window_info("rain", RAIN_START, RAIN_END)

              city                     variable  first_day   last_day
           Kolkata temperature_2m_previous_day1 2022-01-01 2025-12-31
           Kolkata temperature_2m_previous_day7 2022-01-01 2025-12-31
           Kolkata  precipitation_previous_day1 2024-01-20 2025-12-31
           Kolkata  precipitation_previous_day7 2024-01-26 2025-12-31
Thiruvananthapuram temperature_2m_previous_day1 2022-01-01 2025-12-31
Thiruvananthapuram temperature_2m_previous_day7 2022-01-01 2025-12-31
Thiruvananthapuram  precipitation_previous_day1 2024-01-20 2025-12-31
Thiruvananthapuram  precipitation_previous_day7 2024-01-26 2025-12-31
     Visakhapatnam temperature_2m_previous_day1 2022-01-01 2025-12-31
     Visakhapatnam temperature_2m_previous_day7 2022-01-01 2025-12-31
     Visakhapatnam  precipitation_previous_day1 2024-01-20 2025-12-31
     Visakhapatnam  precipitation_previous_day7 2024-01-26 2025-12-31

last station record:
city                element
Kolkata             PRCP       2025-08-2

In [18]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nFiles inside interim folder:")

folder = Path("interim")

if folder.exists():
    for file in sorted(folder.iterdir()):
        print(file.name)
else:
    print("Interim folder does not exist!")

Current working directory:
c:\Users\Hari Kaeilash G\OneDrive\DCV_project_final

Files inside interim folder:
api_check_2026-11-05.csv
data_dictionary_draft_2026-11-05.csv
noaa_ghcnd_3stations_2026-11-05.csv
noaa_ghcnd_3stations_2026-11-05.parquet
openmeteo_gfs_seamless_hourly_2026-11-05.csv
openmeteo_gfs_seamless_hourly_2026-11-05.parquet
raw_manifest_2026-11-05.csv
sources_2026-11-05.csv


In [19]:
from pathlib import Path

print("Current directory:", Path.cwd())

print("\nAll Parquet files:")

for file in Path(".").rglob("*.parquet"):
    print(file)

Current directory: c:\Users\Hari Kaeilash G\OneDrive\DCV_project_final

All Parquet files:
interim\noaa_ghcnd_3stations_2026-11-05.parquet
interim\openmeteo_gfs_seamless_hourly_2026-11-05.parquet
